# Fixed Thresholding (TP 1 — Exercise 3)

## Assignment Objectives
Binarize the water-meter image `tp1_tp2_compteur_water_meter.jpg` with the **fixed threshold** $T = 80$:
- pixels with gray level $> 80 \rightarrow 255$ (white),
- pixels with gray level $\le 80 \rightarrow 0$ (black).

Then compare the OpenCV implementation (`imgb1`) with our from-scratch
algorithm (`imgb2`), save `imgb1` to `TAI/outputs/`, and quantify the
equivalence.

## Mathematical Principles & Intuition
Fixed (global) thresholding maps every pixel through the step function
$$\mathrm{imgb}(y,x) = 255 \cdot \mathbb{1}\{I(y,x) > T\}$$
Splitting the histogram at $T$ separates *foreground* (bright) from
*background* (dark) whenever the two intensity populations are reasonably
bimodal. The boundary convention matters: OpenCV's `THRESH_BINARY` uses
strict `$>$`, so $I(y,x) = T$ itself becomes black.


---
## Step 0: Portable Environment Bootstrap (Local IDE / Jupyter + Google Colab)

### What This Cell Does
- Detects Google Colab (`google.colab` in `sys.modules` or `COLAB_RELEASE_TAG`); on Colab it clones `APSD_TAI_LAB` into `/content` (unless already mounted from Drive), so the notebook works even when opened directly from GitHub.
- Locally, it walks upward from the current working directory to the repository root (marker: `README.md` + `APDS/` + `TAI/`) and registers it on `sys.path` — the notebook therefore runs identically from the repo root, from `TAI/…/notebooks/`, or from any sub-folder.
- Verifies `opencv-contrib-python` (checking `cv2.ximgproc`) and installs it only when missing.
- Calls `bootstrap()` from `TAI.utils.env_io`, which resolves `TAI/images/` (the single consolidated, SHA-256 deduplicated dataset) and `TAI/outputs/` (runtime artifacts, never committed).

### Why It Is Needed
- `cv2.imshow()` opens blocking GUI windows and freezes Jupyter kernels — everything here renders through inline Matplotlib instead.
- No hardcoded absolute paths (like `/home/...`): every path is derived from the discovered repository root, so the same cell works on any machine, in any IDE, and on Colab.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Vtheonly/APSD_TAI_LAB.git"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    # Google Colab: clone the repository (unless already present, e.g. on Drive)
    _repo = Path("/content/APSD_TAI_LAB")
    if not (_repo / "TAI" / "utils").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True
        )
else:
    # Local IDE / Jupyter: walk upward to the repository root
    # (identified by README.md + APDS/ + TAI/), whatever the launch directory.
    _cwd = Path.cwd().resolve()
    _repo = next(
        (
            c
            for c in (_cwd, *_cwd.parents)
            if (c / "README.md").is_file() and (c / "APDS").is_dir() and (c / "TAI").is_dir()
        ),
        None,
    )
    if _repo is None:
        raise FileNotFoundError(
            f"APSD_TAI_LAB repository root not found above {_cwd}. "
            "Launch Jupyter from inside the cloned repository."
        )
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))

# Make sure opencv-contrib-python is used (cv2.ximgproc) -- install only if missing
try:
    import cv2

    _has_ximgproc = hasattr(cv2, "ximgproc")
except ImportError:
    _has_ximgproc = False
if not _has_ximgproc:
    %pip install -q opencv-contrib-python
    import cv2

from TAI.utils.env_io import bootstrap, load_image, show_images, plot_image_with_hist

ROOT_DIR, IMG_DIR, OUT_DIR = bootstrap()

import math
import time

import matplotlib.pyplot as plt
import numpy as np

print(f"Repository root : {ROOT_DIR}")
print(f"Images folder   : {IMG_DIR} ({len(list(IMG_DIR.iterdir()))} images)")
print(f"Outputs folder  : {OUT_DIR}")
print(f"OpenCV          : {cv2.__version__} | cv2.ximgproc available: {hasattr(cv2, 'ximgproc')}")


---
## Step 1: OpenCV vs. From-Scratch Thresholding
`seuillage_fixe_perso` (from `TAI.utils.basics`) is an explicit per-pixel loop
so the traversal stays visible; `TAI.utils` keeps one canonical, unit-tested
implementation shared by every notebook.


In [ ]:
from TAI.utils.basics import seuillage_fixe_perso

img_gray = load_image("tp1_tp2_compteur_water_meter.jpg", cv2.IMREAD_GRAYSCALE)
SEUIL = 80

# 1) OpenCV reference
_, imgb1 = cv2.threshold(img_gray, SEUIL, 255, cv2.THRESH_BINARY)

# 2) From-scratch implementation (explicit loops inside TAI.utils.basics)
imgb2 = seuillage_fixe_perso(img_gray, SEUIL)

identiques = np.array_equal(imgb1, imgb2)
print(f"OpenCV and from-scratch strictly identical : {identiques}")

out_path = OUT_DIR / "tp1_exo3_compteur_binary_imgb1.jpg"
cv2.imwrite(str(out_path), imgb1)
print(f"Binary image saved to: {out_path}")

import time
t0 = time.perf_counter()
seuillage_fixe_perso(img_gray, SEUIL)
dt_loop = (time.perf_counter() - t0) * 1000
t0 = time.perf_counter()
cv2.threshold(img_gray, SEUIL, 255, cv2.THRESH_BINARY)
dt_cv = (time.perf_counter() - t0) * 1000
print(f"Runtime: OpenCV {dt_cv:.3f} ms | Python loop {dt_loop:.1f} ms ({dt_loop / max(dt_cv, 1e-9):.0f}x slower but pedagogically explicit)")


In [ ]:
diff_map = np.abs(imgb1.astype(np.int16) - imgb2.astype(np.int16)).astype(np.uint8)
white_ratio = float(np.mean(imgb1 == 255)) * 100

# Histogram with the threshold position
hist = cv2.calcHist([img_gray], [0], None, [256], [0, 256]).ravel()

fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))
axes[0].imshow(img_gray, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original (grayscale)")
axes[0].axis("off")
axes[1].plot(hist, color="black")
axes[1].axvline(SEUIL, color="red", linestyle="--", linewidth=2, label=f"T = {SEUIL}")
axes[1].set_title("Histogram & fixed threshold")
axes[1].set_xlabel("Intensity"); axes[1].set_ylabel("Pixel count")
axes[1].set_xlim([0, 255]); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[2].imshow(imgb1, cmap="gray", vmin=0, vmax=255)
axes[2].set_title(f"imgb1: OpenCV binary ({white_ratio:.1f}% white)")
axes[2].axis("off")
axes[3].imshow(diff_map, cmap="gray")
axes[3].set_title("|imgb1 - imgb2| (all zeros)")
axes[3].axis("off")
plt.tight_layout()
plt.show()

show_images([
    ("imgb1 - OpenCV", imgb1),
    ("imgb2 - From scratch", imgb2),
    ("Difference map", diff_map),
], cols=3)


## Analysis & Key Takeaways
- `seuillage_fixe_perso` matches `cv2.threshold(..., THRESH_BINARY)` **bit-for-bit** (the difference map is uniformly zero) — the strict `$>$` convention is replicated exactly.
- The Python loop is ~100x slower than OpenCV's SIMD implementation but makes the pixel traversal explicit; production code should use `cv2.threshold` or a vectorized `np.where`.
- A *single* fixed $T=80$ works here because the water-meter background and the meter digits form two clearly separated histogram modes; uneven illumination would call for adaptive thresholding (see the morphology lab).
